In [1]:
from pathlib import Path
import shutil
import random

In [2]:
base = Path("../datasets")

manual = base / "dataset_manual"
source_images = base / "dataset" / "Original" / "OriginalMaster"

target = base / "yolo_manual"

for folder in [
    target/"images/train",
    target/"images/val",
    target/"labels/train",
    target/"labels/val"
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )

In [3]:
label_files = list(
    (manual/"obj_train_data").glob("*.txt")
)

print(
    "Labels:",
    len(label_files)
)

Labels: 200


In [4]:
random.seed(42)

random.shuffle(label_files)

split = int(
    len(label_files)*0.8
)

train_labels = label_files[:split]
val_labels = label_files[split:]

print(
    len(train_labels),
    len(val_labels)
)

160 40


In [5]:
def copy_sample(
    label_path,
    split_name
):

    image_name = (
        label_path.stem + ".jpg"
    )

    image_path = (
        source_images /
        image_name
    )

    shutil.copy(
        image_path,
        target /
        "images" /
        split_name /
        image_name
    )

    shutil.copy(
        label_path,
        target /
        "labels" /
        split_name /
        label_path.name
    )

In [6]:
for lbl in train_labels:
    copy_sample(
        lbl,
        "train"
    )

for lbl in val_labels:
    copy_sample(
        lbl,
        "val"
    )

print("Done.")

Done.


In [7]:
yaml_text = """
path: ../datasets/yolo_manual

train: images/train
val: images/val

names:
  0: muzzle
"""

with open(
    target/"dataset.yaml",
    "w"
) as f:

    f.write(
        yaml_text
    )

print(
    "dataset.yaml created"
)

dataset.yaml created


In [8]:
print(
    len(
        list(
            (target/"images/train").glob("*.jpg")
        )
    )
)

print(
    len(
        list(
            (target/"labels/train").glob("*.txt")
        )
    )
)

print(
    len(
        list(
            (target/"images/val").glob("*.jpg")
        )
    )
)

print(
    len(
        list(
            (target/"labels/val").glob("*.txt")
        )
    )
)

160
160
40
40


In [9]:
from ultralytics import YOLO

model = YOLO(
    "yolov8n.pt"
)

results = model.train(
    data="../datasets/yolo_manual/dataset.yaml",

    epochs=50,

    imgsz=640,

    batch=8,

    device=0,

    workers=4,

    project="../models",

    name="muzzle_detector_manual"
)

New https://pypi.org/project/ultralytics/8.4.57 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.54  Python-3.11.15 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../datasets/yolo_manual/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0

In [10]:
from ultralytics import YOLO

model = YOLO(
    "../notebooks/runs/models/muzzle_detector_manual/weights/best.pt"
)

results = model.predict(
    source="../datasets/cow_1.jpg",
    conf=0.25
)

results[0].show()


image 1/1 d:\VS Code\Muzzle_Net\notebooks\..\datasets\cow_1.jpg: 448x640 1 muzzle, 71.3ms
Speed: 3.5ms preprocess, 71.3ms inference, 3.3ms postprocess per image at shape (1, 3, 448, 640)
